In [9]:
import os
os.chdir(r"C:\Users\mpree\cfpb-classifier")
print(os.getcwd())
print(os.path.exists("complaints.csv")) 

C:\Users\mpree\cfpb-classifier
True


In [13]:
import pandas as pd
df = pd.read_csv("complaints.csv")
print(df.columns.tolist())   # confirm column names
print(df.shape)              # confirm row count
print(df['product'].value_counts())  # see label distribution

['Unnamed: 0', 'product', 'narrative']
(162421, 3)
product
credit_reporting       91179
debt_collection        23150
mortgages_and_loans    18990
credit_card            15566
retail_banking         13536
Name: count, dtype: int64


In [17]:
import pandas as pd
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def load_and_clean(filepath="complaints.csv", n_classes=8):
    df = pd.read_csv(filepath)
    print(f"Raw rows loaded: {len(df)}")
    print(f"Columns found: {list(df.columns)}")

    df = df[['narrative', 'product']].copy()
    df.columns = ['text', 'label']

    # Drop empty
    df = df[df['text'].notna()]
    df = df[df['text'].str.strip() != '']
    df = df[df['label'].notna()]

    # ✅ simple sample — no groupby needed
    if len(df) > 50000:
        df = df.sample(50000, random_state=42).reset_index(drop=True)
        print(f"Sampled down to: {len(df)} rows")

    # Keep top N categories
    top_categories = df['label'].value_counts().nlargest(n_classes).index
    df = df[df['label'].isin(top_categories)]

    # Clean text
    def clean(text):
        text = str(text).lower()
        text = re.sub(r'x{2,}', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['text'] = df['text'].apply(clean)
    df = df[df['text'].str.split().str.len() > 10]

    # Encode labels
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])
    num_classes = len(le.classes_)

    print(f"\nClasses ({num_classes}): {list(le.classes_)}")
    print(f"Final dataset size: {len(df)}")
    print(f"Label distribution:\n{df['label'].value_counts()}\n")

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        df['text'], df['label'],
        test_size=0.2, random_state=42, stratify=df['label']
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=0.1, random_state=42
    )

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test, num_classes, le

# call it
X_train, X_val, X_test, y_train, y_val, y_test, num_classes, le = load_and_clean()

Raw rows loaded: 162421
Columns found: ['Unnamed: 0', 'product', 'narrative']
Sampled down to: 50000 rows

Classes (5): ['credit_card', 'credit_reporting', 'debt_collection', 'mortgages_and_loans', 'retail_banking']
Final dataset size: 46788
Label distribution:
label
1    25669
2     6813
3     5687
0     4712
4     3907
Name: count, dtype: int64

Train: 33687 | Val: 3743 | Test: 9358


In [3]:
pip install imblearn


   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ------------- -------------------------- 1/3 [imbalanced-learn]
   ---------------------------------------- 3/3 [imblearn]

Note: you may need to restart the kernel to use updated packages.


In [19]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

def load_and_clean(filepath="complaints.csv", n_classes=8):
    df = pd.read_csv(filepath)
    print(f"Raw rows loaded: {len(df)}")

    df = df[['narrative', 'product']].copy()
    df.columns = ['text', 'label']
    df = df[df['text'].notna()]
    df = df[df['text'].str.strip() != '']
    df = df[df['label'].notna()]
    top_categories = df['label'].value_counts().nlargest(n_classes).index
    df = df[df['label'].isin(top_categories)]
    def clean(text):
        text = str(text).lower()
        text = re.sub(r'x{2,}', '', text)
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['text'] = df['text'].apply(clean)
    df = df[df['text'].str.split().str.len() > 10]

    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])
    num_classes = len(le.classes_)

    print(f"\nBefore balancing:")
    print(df['label'].value_counts())

    under_strategy = {}
    for cls in df['label'].unique():
        count = df['label'].value_counts()[cls]   
        under_strategy[cls] = min(count, 10000)   

    under = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)  
    X_res, y_res = under.fit_resample(
        df['text'].values.reshape(-1, 1),
        df['label']
    )

    over_strategy = {}
    unique, counts = np.unique(y_res, return_counts=True)
    for cls, count in zip(unique, counts):
        if count < 8000:
            over_strategy[cls] = 8000

    over = RandomOverSampler(sampling_strategy=over_strategy, random_state=42)
    X_bal, y_bal = over.fit_resample(X_res, y_res)

    df_balanced = pd.DataFrame({
        'text':  X_bal.flatten(),
        'label': y_bal
    })

    print(f"\nAfter balancing:")
    print(df_balanced['label'].value_counts())
    print(f"\nClasses ({num_classes}): {list(le.classes_)}")
    print(f"Final dataset size: {len(df_balanced)}")

    X_train, X_test, y_train, y_test = train_test_split(
        df_balanced['text'], df_balanced['label'],
        test_size=0.2, random_state=42, stratify=df_balanced['label']
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=0.1, random_state=42
    )

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, num_classes, le

X_train, X_val, X_test, y_train, y_val, y_test, num_classes, le = load_and_clean()

Raw rows loaded: 162421

Before balancing:
label
1    83668
2    21870
3    18669
0    15255
4    12570
Name: count, dtype: int64

After balancing:
label
0    10000
1    10000
2    10000
3    10000
4    10000
Name: count, dtype: int64

Classes (5): ['credit_card', 'credit_reporting', 'debt_collection', 'mortgages_and_loans', 'retail_banking']
Final dataset size: 50000
Train: 36000 | Val: 4000 | Test: 10000


In [5]:
import os
os.makedirs("saved_models", exist_ok=True)  # creates folder if it doesn't exist
print("saved_models folder created!")

saved_models folder created!


In [21]:
import pickle

with open('saved_models/preprocessed_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_val':   X_val,
        'X_test':  X_test,
        'y_train': y_train,
        'y_val':   y_val,
        'y_test':  y_test,
        'num_classes': num_classes,
        'le':      le
    }, f)

print("Saved! Other notebooks can load this directly.")

Saved! Other notebooks can load this directly.
